# Impulse response coding

This version mimicks the script version. Executed from top to bottom.

This version has stopped being developed on 20260720, after reaching the end plotting the stacked spectra and the impulse response

In [ ]:
import numpy as np
from dataclasses import dataclass, field
from astropy.table import Table
from astropy.coordinates import SpectralCoord
import astropy.units as u
import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnchoredText
from specutils import Spectrum
from specutils.manipulation import LinearInterpolatedResampler
from specutils.manipulation import FluxConservingResampler
from typing import Optional

In [ ]:
# Aux functions
def gaussian_profile(vel_array, intensity, velocity, width):
    sigma = width / (2 * np.sqrt(2 * np.log(2)))
    return intensity * np.exp(-(vel_array - velocity)**2 / (2 * sigma**2))

In [ ]:
@dataclass
class LineSpectrum:
    observed: Spectrum
    model: Spectrum

    # Define your measurement fields here (defaulting to None)
    noise: Optional[u.Quantity] = None
    snr: Optional[float] = None
    line_center: Optional[u.Quantity] = None

    # This safely initializes an empty dictionary for every new instance
    meta: dict = field(default_factory=dict)

    def to_vel(self):
        velocity_line = LineSpectrum(
            observed=Spectrum(
                spectral_axis=self.observed.velocity.copy(),
                flux=self.observed.flux.copy(),
                rest_value=self.observed.rest_value,
                velocity_convention=self.observed.velocity_convention,
            ),
            model=Spectrum(
                spectral_axis=self.model.velocity.copy(),
                flux=self.model.flux.copy(),
                rest_value=self.model.rest_value,
                velocity_convention=self.observed.velocity_convention,
            ),
            meta=self.meta.copy(),
        )
        return velocity_line
    
    def reverse(self):
        reversed_line = LineSpectrum(
            observed=Spectrum(
                spectral_axis=self.observed.spectral_axis[::-1].copy(),
                flux=self.observed.flux[::-1].copy(),
                rest_value=self.observed.rest_value,
                velocity_convention=self.observed.velocity_convention,
            ),
            model=Spectrum(
                spectral_axis=self.model.spectral_axis[::-1].copy(),
                flux=self.model.flux[::-1].copy(),
                rest_value=self.model.rest_value,
                velocity_convention=self.observed.velocity_convention,
            ),
            meta=self.meta.copy(),
        )
        return reversed_line
    
    def resample(self, resampler, new_axis):

        # Change units of spectra to new_axis units

        resampled_observed = resampler(self.observed, new_axis)
        resampled_model = resampler(self.model, new_axis)
        
        # First velocity convention has to be set, then the rest_value setter.
        if hasattr(self.observed, 'velocity_convention'):
            resampled_observed = resampled_observed.with_velocity_convention(
                self.observed.velocity_convention)
            resampled_model = resampled_model.with_velocity_convention(
                self.model.velocity_convention)
        if hasattr(self.observed, 'rest_value'):
            resampled_observed.rest_value = self.observed.rest_value
            resampled_model.rest_value = self.model.rest_value

        return LineSpectrum(
            observed=resampled_observed,
            model=resampled_model,
            meta=self.meta.copy(),
        )

    @property
    def spectral_axis(self):
        return self.observed.spectral_axis
    
    @property
    def flux(self):
        return self.observed.flux
    
    @property
    def rest_value(self):
        return self.observed.rest_value

# User input

In [ ]:
# User input

# Set SLIM data
molecule = "PO,v=0"  # Formula string from MADCUBA
slim_freq_unit = u.MHz
slim_filename = "cross-corr-dats/slim_table_region_10.csv"
slim_filename = "cross-corr-dats/SLIM-table-SO.csv"

# Set spectra data (can be multiple files)
spectra_filenames = [
    "cross-corr-dats/01_IRAS_16293-2422_spw27.dat",
    "cross-corr-dats/01_IRAS_16293-2422_spw29.dat",
    # "cross-corr-dats/POband7-obsfreq.dat",
]
spectra_freq_unit = u.GHz
spectra_flux_unit = u.Jy

# Freq list
# Here the user can select which transitions to use for the stacking.
# If set to 'all', the program uses every transition found within the spectra
# files
stacking_freqs = [
    152656.979,
    152680.282,
    152855.454,
    152888.128,
    283586.8296,
    283593.1805,
    283777.603,
    283785.4132,
]
stacking_freqs_unit = u.MHz


# Stacking parameters

stacking_weighting = 'equal'    # 'equal' or 'sigma'

stacking_window_params = {
    'min_vel': -60,    # min velocity (km/s) of the window around each line
    'max_vel': 60,    # max velocity (km/s) of the window around each line
}

baseline_window_params = {
    'min_vel': -1,    # min velocity (km/s) of the window around each line
    'max_vel': 1,    # max velocity (km/s) of the window around each line
}

# Resampling parameters

# If False, the program uses the velocity array for the first line and
# custom_velocity_axes_params are ignored:
use_custom_velocity_axes = False
# If use_custom_velocity_axes is True:
custom_velocity_axes_params = {
    'ref_vel': 0,    # reference velocity to resample velocity axis
    'channel_width': 2    # channel width in velocity (km/s)
}

resample_method = 'flux-conserving'  # 'linear' (default) or 'flux-conserving' (CLASS)

# Logging
view_each_transition = True

# Data preparation

Read data and prepare the observed spectra and models for the select transitions

In [ ]:
# Check that all freq units are both the same physical type (Frequency)
if (spectra_freq_unit.physical_type != 'frequency'
    or stacking_freqs_unit.physical_type != 'frequency'
    or slim_freq_unit.physical_type != 'frequency'):
    raise TypeError("Freq variables must be of physical type 'frequency'!")

# SLIM and Spectra preparation

slim_table = Table.read(slim_filename, format='csv', delimiter=",")
slim_table["Frequency"].unit = slim_freq_unit

# Molecule filter
molecule_filter = (slim_table["Formula"] == molecule)

# Frequency filter
# Using Spectra class. Stores Spectrum anD Table just in case
spectra_tables = []
spectra = []
for filename in spectra_filenames:
    spectrum_table = Table.read(filename, format="ascii")
    spectra_tables.append(spectrum_table)
    spectra.append(
        Spectrum(
            spectral_axis=spectrum_table.columns[0].data * spectra_freq_unit,
            flux=spectrum_table.columns[1].data * spectra_flux_unit,
        )
    )
freq_range_filter = np.zeros(len(slim_table), dtype=bool)
for spectrum in spectra:
    spec_freq = spectrum.spectral_axis
    freq_range_filter |= (
        (slim_table["Frequency"] > spec_freq.min()) &
        (slim_table["Frequency"] < spec_freq.max())
    )
slim_table_mol_freq = slim_table[molecule_filter & freq_range_filter]

# Selected transitions filter
selected_freqs = stacking_freqs * stacking_freqs_unit
table_freqs = slim_table_mol_freq["Frequency"].to(u.GHz).value
selected_freqs_filter = np.zeros(len(table_freqs), dtype=bool)
for f in selected_freqs.to(u.GHz).value:
    selected_freqs_filter |= np.isclose(
        table_freqs,
        f,
        rtol=0,
        atol=1e-6,  # Hz-precision when comparing MHz
    )
slim_table_stacking = slim_table_mol_freq[selected_freqs_filter]
slim_table_stacking

In [ ]:
group_col = "Transition" if "Transition" in slim_table_stacking.colnames else "Frequency"
slim_table_stacking_grouped = slim_table_stacking.group_by(group_col)

lines = []
for key, group in zip(slim_table_stacking_grouped.groups.keys,
                      slim_table_stacking_grouped.groups):
    # Here group is each transition table of the molecule

    # Rest frequency of this transition
    rest_freq = group["Frequency"].quantity[0]

    # Check if all rows are the same transition
    group_freqs = group["Frequency"].quantity
    if "Transition" in group.colnames:
        if not np.all(group["Transition"] == group["Transition"][0]):
            raise ValueError(
                f"Transitions check failed: they differ between velocity components"
            )
    elif not np.allclose(
        group_freqs.to_value(u.MHz), group_freqs[1].to_value(u.MHz),
        rtol=1e-12, atol=1e-12):  # This has precision up to the Hz on MHz arrays
            raise ValueError(
                f"Transitions check failed: they differ between velocity components"
            )

    component_lines = np.column_stack([
        group["Intensity"],
        group["Velocity"],
        group["Width"],
    ])

    for spectrum in spectra:

        # Check if transition is inside this spectrum
        if not (
            spectrum.spectral_axis.min() <= rest_freq <= spectrum.spectral_axis.max()
        ):
            continue

        found = True

        # Create models

        # Velocity axis for this spectrum
        spec_vel_value = spectrum.spectral_axis.to(
            u.km / u.s,
            doppler_rest=rest_freq,
            doppler_convention="radio",
        ).value
        # Observed flux
        spectrum_flux_value = spectrum.flux.value
        # Create Gaussian model
        model = np.zeros_like(spec_vel_value)

        for intensity, vel, width in component_lines:
            model += gaussian_profile(spec_vel_value, intensity, vel, width)

        # Create the LineSpectrum objects
        observed_spectrum = Spectrum(
            spectral_axis=SpectralCoord(
                spectrum.spectral_axis,
                doppler_rest=rest_freq,
                doppler_convention="radio",
            ),
            flux=spectrum.flux,
            rest_value=rest_freq,
            velocity_convention="radio",
        )
        model_spectrum = Spectrum(
            spectral_axis=SpectralCoord(
                spectrum.spectral_axis,
                doppler_rest=rest_freq,
                doppler_convention="radio",
            ),
            flux=model * spectra_flux_unit,
            rest_value=rest_freq,
            velocity_convention="radio",
        )
        line = LineSpectrum(
            observed=observed_spectrum,
            model=model_spectrum,
        )
        lines.append(line)

    if not found:
        raise RuntimeError(
            f"Transition at {rest_freq} was not found in any selected spectrum."
        )

# Plot selected transitions

In [ ]:
# Plot

if view_each_transition:

    figsize = (6, 8)
    fig, axes = plt.subplots(4, 1, figsize=figsize)

    for ax, line in zip(axes, lines):

        ax.plot(
            line.spectral_axis.to(u.km/u.s),
            line.flux,
            drawstyle="steps-mid",
            color="black",
            alpha=0.8,
        )
        ax.plot(
            line.spectral_axis.to(u.km/u.s),
            line.model.flux,
            drawstyle="steps-mid",
            color="red",
        )

        # ax.axvline(0.0, color="k", ls="--", alpha=0.5)

        ax.set_ylabel("Intensity")
        ax.set_xlabel("Velocity [km/s]")
        ax.set_title("SLIM generated model over observed spectrum")
        ax.text(
            0.02,
            0.98,
            f"{line.rest_value}",
            ha="left",
            va="top",
            transform=ax.transAxes,
        )

        ax.set_xlim(-40, 40)
        ax.set_ylim(-1.5 * model.max(), 2 * model.max())

    plt.tight_layout()
    # plt.savefig(f"{line.rest_value}-mult.png", dpi=300)
    plt.show()

# Stack selected transitions

In [ ]:
# Prepare common velocity axis

v_min = stacking_window_params['min_vel']
v_max = stacking_window_params['max_vel']

# Prepare slices common velocity axes
if use_custom_velocity_axes == True:
    # Custom velocity axis with vref in the middle of its channel
    dv = custom_velocity_axes_params['channel_width']
    v_ref = custom_velocity_axes_params["ref_vel"]
    # Calculate steps relative to v_ref to each side
    # Using ceil ensures we fully cover the requested window boundaries
    n_steps_left = int(np.ceil((v_ref - v_min) / dv))
    n_steps_right = int(np.ceil((v_max - v_ref) / dv))
    # Generate step integers relative to the reference channel
    steps = np.arange(-n_steps_left, n_steps_right + 1)
    # Get the common velocity axis anchored at v_ref
    common_velocity = v_ref + (steps * dv)
else:
    # Get common velocity as the velocity axes of the first line 
    # spectra_sc_template = SpectralCoord(spectra_obsfreq, unit=x_axis_unit,
    #                                     doppler_rest=line_rest_freqs_quant[0],
    #                                     doppler_convention='radio',)
    common_velocity_value = lines[0].spectral_axis.to(u.km / u.s).value
    # common_velocity_value = spectra_sc_template.to(u.km / u.s).value
    slice_mask = ((common_velocity_value >= v_min) &
                  (common_velocity_value <= v_max))
    common_velocity = common_velocity_value[slice_mask]
    # If the axis was in frequency, we need to swap its order so that the velocity
    # array is incremental
    if common_velocity[0] > common_velocity[-1]:
        common_velocity = common_velocity[::-1]
    
resample_velocity = common_velocity * u.km / u.s

In [ ]:
# Resample every line to the same velocity axis

linear_resampler = LinearInterpolatedResampler(
    extrapolation_treatment='nan_fill')
flux_conserving_resampler = FluxConservingResampler(
    extrapolation_treatment='nan_fill')

lines_resampled = []
# Slice spectra around each line and resample them to the common velocity axis
for line in lines:

    v_slice = line.spectral_axis.to(u.km / u.s).value.copy()
    # v_slice = current_line.spectral_axis.to_value(u.km / u.s)

    # Interpolate the data onto the common velocity grid slice
    # Ensure the velocity axis is strictly increasing for resampling
    # Ensure increasing velocity axis
    if (len(v_slice) > 1 and v_slice[0] > v_slice[-1]):
        current_line = line.reverse()
    else:
        current_line = line

    # Resample line
    if resample_method == 'linear':
        line_resampled = current_line.to_vel().resample(
            linear_resampler, resample_velocity)
    elif resample_method == 'flux-conserving':
        line_resampled = current_line.to_vel().resample(
            flux_conserving_resampler, resample_velocity)

    # Store resampled lines
    lines_resampled.append(line_resampled)

In [ ]:
# Stack the lines into just one LineSpectrum object

lines_resampled_baselined = []
for line in lines_resampled:

    f_slice = line.observed.flux.value.copy()
    v_slice = line.observed.spectral_axis.to(u.km / u.s).value.copy()

    # Baseline each slice before stacking to allow sigma based weighting
    # Define your window boundaries for the baseline
    base_min = stacking_window_params['min_vel']
    base_max = stacking_window_params['max_vel']
    # Create a mask specifically matching the baseline region on the common grid
    baseline_mask = (v_slice >= base_min) & (v_slice <= base_max)
    # Calculate the average background level inside this window
    # (Using np.mean here since we want the unweighted baseline value)
    baseline_level = np.mean(f_slice[baseline_mask])
    # Level the spectrum by subtracting the baseline from the whole slice
    f_slice -= baseline_level

    line_resampled_baselined = LineSpectrum(
        observed=Spectrum(
            spectral_axis=line.spectral_axis.copy(),
            flux=f_slice * line.flux.unit,
            rest_value=line.rest_value.copy(),
            velocity_convention=line.observed.velocity_convention,
        ),
        model=Spectrum(
            spectral_axis=line.spectral_axis.copy(),
            flux=line.model.flux.copy(),
            rest_value=line.rest_value.copy(),
            velocity_convention=line.model.velocity_convention,
        ),
        meta=line.meta.copy(),
    )
    lines_resampled_baselined.append(line_resampled_baselined)

In [ ]:
# Stack the spectra and model

obs_slices = [line.observed.flux.to(spectra_flux_unit).value.copy() for line in lines_resampled_baselined]
model_slices = [line.model.flux.to(spectra_flux_unit).value.copy() for line in lines_resampled_baselined]

if stacking_weighting == 'equal':
    weights = None
elif stacking_weighting == 'sigma':
    std_devs = [np.std(slice) for slice in obs_slices]  # estimate noise from slice
    weights = 1.0 / np.array(std_devs)**2
else: weights = None

stacked_observed = np.average(obs_slices, axis=0, weights=weights)
stacked_model = np.average(model_slices, axis=0, weights=weights)

template_line = lines_resampled_baselined[0]
line_stacked = LineSpectrum(
    observed=Spectrum(
        spectral_axis=line.spectral_axis.copy(),
        flux=stacked_observed * template_line.observed.unit,
        rest_value=template_line.rest_value.copy(),
        velocity_convention=template_line.observed.velocity_convention,
    ),
    model=Spectrum(
        spectral_axis=line.spectral_axis.copy(),
        flux=stacked_model * template_line.observed.unit,
        rest_value=template_line.rest_value.copy(),
        velocity_convention=template_line.model.velocity_convention,
    ),
)

# Correlation

In [ ]:
# Compute cross-correlation

# Check same number of channels in all arrays:
check_length_arrays = np.array([
    line_stacked.observed.flux.value,
    line_stacked.observed.spectral_axis.value,
    line_stacked.model.flux.value,
    line_stacked.model.spectral_axis.value,
])
all_same_length = len(set(len(arr) for arr in check_length_arrays)) == 1
n_chan = len(line_stacked.observed.flux.value) if all_same_length else None

# Extract the physical velocity axis in km/s and model flux arrays
v_axis = line_stacked.observed.spectral_axis.to(u.km / u.s).value
model_flux = line_stacked.model.flux.value
# Calculate the intensity-weighted center (centroid) of the model line (e.g., -3.1 km/s)
v_center = np.sum(v_axis * model_flux) / np.sum(model_flux)
# Get the relative channel spacing steps measured from the middle of the array
relative_offsets = v_axis - v_axis[n_chan // 2]
lags = relative_offsets
# Anchor the lag axis to the centroid value instead of 0
lags_shifted = relative_offsets + v_center


## Scipy FFT cross-correlation
F_data = np.fft.fft(line_stacked.observed.flux.value)
F_model = np.fft.fft(line_stacked.model.flux.value)
corr = np.fft.ifft(F_data * np.conj(F_model)).real

# Move zero-lag to the center of the array for the correlation data
corr = np.fft.fftshift(corr)

# Normalize correlation to S/N units
corr /= np.std(corr)
# corr /= (np.std(corr)**2)

# Results

In [ ]:
# Plot results
main_fig, main_fig_axes = plt.subplots(1, 2, figsize=(12,5))


main_fig_axes[0].plot(line_stacked.spectral_axis, line_stacked.flux, drawstyle='steps-mid',
                color='black', lw=2)
main_fig_axes[0].plot(line_stacked.spectral_axis, line_stacked.model.flux,
                color='red', lw=2)

if use_custom_velocity_axes:
    main_fig_axes[0].axvline(v_ref, color='red', linestyle='--', alpha=0.7)
main_fig_axes[0].grid(True, alpha=0.3)

main_fig_axes[0].set_xlabel('Velocity (km/s)')
main_fig_axes[0].set_ylabel('Flux / Intensity')
main_fig_axes[0].set_title('Stacked Molecular Emission Lines')
# main_fig_axes[0].set_xlim(-20, 20)
main_fig_axes[0].set_ylim(-0.005, 0.0052)

main_fig_axes[1].plot(lags, corr, color='black')
main_fig_axes[1].set_xlabel("Velocity shift (km/s)")
main_fig_axes[1].set_ylabel("S/N")
main_fig_axes[1].set_title("Impulse response")
# main_fig_axes[1].set_xlim(-20, 20)

peak_shift = lags[np.argmax(corr)]
# print(f"Peak correlation S/N            = {peak_sn:.2f}")
print(f"Recovered shift               = {peak_shift:.3f} km/s")
print(f"Line centroid                 = {v_center:.3f} km/s")
# print(f"Detected velocity of first comp = {peak_vel:.3f} km/s")

plt.tight_layout()
plt.show()

---

---